In [ ]:
import pandas as pd
import spacy
from collections import Counter
import numpy as np
import re
import random

random.seed(42)
np.random.seed(42)

# Load spaCy English model
nlp = spacy.load("en_core_web_lg")

In [ ]:
# Load all three model outputs
gpt4o_df = pd.read_csv("/Data/gpt4o_climate.csv")
llama_df = pd.read_csv("/Data/llama3.3_climate.csv")
mistral_df = pd.read_csv("/Data/mistral_large2.1_climate.csv")

In [32]:
# Add model column for identification
gpt4o_df["model"] = "gpt-4o"
llama_df["model"] = "llama-3.3"
mistral_df["model"] = "mistral-large-2.1"

# Combine all
df = pd.concat([gpt4o_df, llama_df, mistral_df], ignore_index=True)

# Ensure age column exists 
if "age" not in df.columns:
    raise ValueError("CSV missing 'age' column required for bias analysis")

In [33]:
# Function to extract nouns and adjectives
def extract_nouns_adjs(text):
    doc = nlp(str(text))
    nouns = [token.lemma_.lower() for token in doc if token.pos_ == "NOUN"]
    adjs = [token.lemma_.lower() for token in doc if token.pos_ == "ADJ"]
    return nouns, adjs

df[["nouns", "adjectives"]] = df["output"].apply(
    lambda x: pd.Series(extract_nouns_adjs(x))
)


In [34]:
def compute_odds_ratio_age(df, pos, target_age):
    """
    Compute Odds Ratio (OR) for words in one age group vs. all others.
    
    Args:
        df: DataFrame with 'age' column and tokenized words in `pos` column (list of words).
        pos: str, either "nouns" or "adjectives"
        target_age: str, the age group to analyze (e.g., "Young Adult (18-24)")
    
    Returns:
        DataFrame with word frequencies and OR scores, sorted by OR.
    """
    # Words from target group
    target_words = [w for words in df[df.age == target_age][pos] for w in words]
    # Words from all other groups
    other_words = [w for words in df[df.age != target_age][pos] for w in words]

    target_counts = Counter(target_words)
    other_counts = Counter(other_words)

    results = []
    all_words = set(target_counts.keys()) | set(other_counts.keys())

    for word in all_words:
        Et = target_counts[word]
        Eo = other_counts[word]

        Et_other = sum(target_counts.values()) - Et
        Eo_other = sum(other_counts.values()) - Eo

        # Add-1 smoothing
        OR = ((Et + 1) / (Et_other + 1)) / ((Eo + 1) / (Eo_other + 1))

        results.append({"word": word, "Et": Et, "Eo": Eo, "OR": OR})

    return pd.DataFrame(results).sort_values("OR", ascending=False)

In [ ]:
# Generate top 10 for nouns and adjectives per model
for model_name, group in df.groupby("model"):
    print(f"\n=== Model: {model_name} ===")
    age_groups = group["age"].unique()

    for age in age_groups:
        print(f"\n=== Age Group: {age} ===")
        
        noun_or = compute_odds_ratio_age(group, pos="nouns", target_age=age)
        adj_or = compute_odds_ratio_age(group, pos="adjectives", target_age=age)

        print("\nTop 10 salient nouns:")
        print(noun_or.head(10)[["word","Et","Eo","OR"]])
        
        print("\nTop 10 salient adjectives:")
        print(adj_or.head(10)[["word","Et","Eo","OR"]])

        print("\nBottom 10 salient nouns:")
        print(noun_or.tail(10)[["word","Et","Eo","OR"]])
        
        print("\nBottom 10 salient adjectives:")
        print(adj_or.tail(10)[["word","Et","Eo","OR"]])

### WEAT Test

In [36]:
# =========================
# 1. Load spaCy embeddings
# =========================
print("Loading spaCy embeddings...")
nlp = spacy.load("en_core_web_lg")  # or en_core_web_lg

def get_vector(word):
    token = nlp.vocab[word]
    if token.has_vector:
        return token.vector
    else:
        return None

def get_vector(word):
    token = nlp.vocab[word]
    if token.has_vector:
        return token.vector
    else:
        return None

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

Loading spaCy embeddings...


In [37]:
def weat_score(target_X, target_Y, attr_A, attr_B):
    def s(w, A, B):
        return np.mean([cosine_similarity(w, a) for a in A]) - np.mean([cosine_similarity(w, b) for b in B])
    
    X_vecs = [get_vector(w) for w in target_X if get_vector(w) is not None]
    Y_vecs = [get_vector(w) for w in target_Y if get_vector(w) is not None]
    A_vecs = [get_vector(w) for w in attr_A if get_vector(w) is not None]
    B_vecs = [get_vector(w) for w in attr_B if get_vector(w) is not None]
    
    score = np.mean([s(x, A_vecs, B_vecs) for x in X_vecs]) - np.mean([s(y, A_vecs, B_vecs) for y in Y_vecs])
    return score

In [38]:
# =========================
# 2. Attribute words
# =========================
## Innovation vs. Tradition
innovation_words = ["creative", "novel", "dynamic", "future", "progressive", "pioneering","growth","exploration","innovation","ambition"]
tradition_words = ["heritage", "custom", "stability", "continuity", "experience", "wisdom", "established", "balance", "settled","legacy","root"]


In [45]:
## Energy vs. Experience
energy_words = [ "vibrant", "active", "fast", "adventurous", "dynamic","wild","power"]
experience_words = ["wise", "seasoned", "knowledgeable", "reliable", "thoughtful", "influential"]

In [40]:
# # # ============================
# # # 2. Define lexicon categories 
# # # ============================

# #### Cuddy, Fiske, & Glick (2008). Stereotype Content Model, orth & Fiske (2012). An Inconvenienced Youth? Ageism and Its Potential Intergenerational Roots. Levy (2009). Stereotype embodiment: A psychosocial approach to aging.
lexicons = {
    # --- Capability / Performance (Competence vs. Incompetence) ---
    "Competence": [
        "competent", "capabl*", "skil*","proficien*", "adept*", "effectiv*", "efficien*", "purposeful",
        "sharp", "quick-witt*", "talent*", "expert*", "savvy", "knowledg*", "reliab*", "professional", "dedicat*",
        "productiv*", "industrious", "resourceful","proactive","activ*","lead*",  "contribut*", "limitless", 
        "competitive", "realistic", "strateg*", "thriv*", "wisdom", "experienced", "value*","endless"
    ],
    "Incompetence": [
        "incompetent", "incapabl*", "unskil*", "inept*", "inefficien*", "ineffectiv*",
        "forgetful", "confus*", "slow-mind*", "clums*", "careless", "unreliabl*", "mistake-prone",
        "struggl*", "error-prone","passive"
    ],

    # --- Warmth / Social Perception (Warmth vs. Coldness) ---
    "Warmth": [
        "warm*", "kind", "caring", "friend*", "support*", "helpful", "generous", "patient", "peace*", "love*", 
        "safe*", "beaut*", "elegan*", "respect*", "thoughtful", "considerat*", "empath*", "compassion*",
         "nurtur*","charm*","enchant*","harmon*", "secur*", "stunning", "graceful", "sensibl*"
    ],
    "Coldness": [
        "cold", "distant", "indifferent", "selfish", "arrogant", "dismissive", "rude",
        "uncaring", "hostil*", "callous", "unfriendly", "harm*"
    ],

    # --- Autonomy / Reliance (Independence vs. Dependence) ---
    "Independence": [
        "independen*", "self-reliant", "selfsufficien*", "autonom*", "capabl*-on-their-own",
        "make-their-own-decision*", "manage-on-their-own", "self-directed", "freedom", "initiat*"
    ],
    "Dependence": [
        "depend*", "reliant", "needy", "fragil*", "frail", "helpless", "vulnerab*", "retire*",
        "need-assist*", "care-dependen*", "requiring-support","reliance", "limit*", "restrict*"
    ],

    # --- Temporal / Ideational Orientation (Progressive vs. Traditional) ---
    "Progressive": [
        "progressiv*", "innovat*", "modern", "forward-looking", "future-oriented", "changemaker", "maker",
        "dynamic", "creative", "adaptiv*", "open-minded", "tech-savvy", "entrepreneur*", "divers*", "global", "impact*"
    ],
    "Traditional": [
        "tradition*", "conservative", "old-fashion*", "heritage", "custom*", "convention*",
        "respect-for-tradition", "time-honor*", "stability-first", "status-quo","stabl*"
    ],


    # --- Vitality / Robustness (Energy vs. Frailty) ---
    "Energy": [
        "energetic", "active", "vibrant", "vigorous", "motivated", "ambiti*","young","youth",
        "eager", "driven", "high-energy", "lively", "force*", "power*", "brave*","wild"
    ],
    "Frailty": [
        "frail", "fragil*", "weak*", "feeble", "delicate", "brittle", "infirm", "decrepit",
        "ailing", "bedridden", "surviv*", "old*","senior*","outdat*"
    ],

    # --- Risk Appraisal / Outlook (Opportunity vs. Risk/Decline) ---
    "Opportunity": [
        "opportunity", "potential", "promis*", "hope*", "bright-future", "optimistic",
        "upside", "room-to-grow", "prospect*", "employ*","grow*","beacon"
    ],
    "Risk": [
        "risk", "declin*", "downturn", "loss", "deteriorat*", "worsen*", "setback",
        "threat*", "danger*", "crisis", "unpredict*"
    ]
}





In [41]:


# 1. Compile all lexicon patterns into one master set of regex
def compile_lexicons(lexicons):
    patterns = []
    for category, words in lexicons.items():
        for word in words:
            # Convert wildcard to regex (e.g., "intelligen*" → "intelligen.*")
            pattern = re.compile("^" + word.replace("*", ".*") + "$", re.IGNORECASE)
            patterns.append(pattern)
    return patterns

# 2. Check if word matches any lexicon pattern
def matches_lexicon(word, lexicon_patterns):
    return any(p.match(word) for p in lexicon_patterns)

# 3. Precompile lexicon patterns once
lexicon_patterns = compile_lexicons(lexicons)

# 4. Initialize dict for storing results
age_word_sets = {}

# 5. Process each model × each age group
age_groups = df["age"].unique()

for model_name, group in df.groupby("model"):
    age_word_sets[model_name] = {}
    
    for age_val in age_groups:
        noun_or = compute_odds_ratio_age(group, pos="nouns", target_age=age_val)
        adj_or = compute_odds_ratio_age(group, pos="adjectives", target_age=age_val)

        # Extract top 10 and bottom 10
        top_nouns = noun_or.head(10)["word"]
        bottom_nouns = noun_or.tail(10)["word"]
        top_adjs = adj_or.head(10)["word"]
        bottom_adjs = adj_or.tail(10)["word"]

        # Filter by lexicon matches
        top_nouns = top_nouns[top_nouns.apply(lambda w: matches_lexicon(w, lexicon_patterns))].reset_index(drop=True)
        bottom_nouns = bottom_nouns[bottom_nouns.apply(lambda w: matches_lexicon(w, lexicon_patterns))].reset_index(drop=True)
        top_adjs = top_adjs[top_adjs.apply(lambda w: matches_lexicon(w, lexicon_patterns))].reset_index(drop=True)
        bottom_adjs = bottom_adjs[bottom_adjs.apply(lambda w: matches_lexicon(w, lexicon_patterns))].reset_index(drop=True)

        # Store results
        age_word_sets[model_name][age_val] = {
            "top_nouns": top_nouns.tolist(),
            "bottom_nouns": bottom_nouns.tolist(),
            "top_adjs": top_adjs.tolist(),
            "bottom_adjs": bottom_adjs.tolist()
        }


In [ ]:
# =========================
# 4. Run WEAT per model × age group
# =========================
results = []

for model_name, age_dict in age_word_sets.items():
    for age_val, words in age_dict.items():
        print(model_name, age_val, words , "\n")
        # Get nouns and adjectives for this age group
        nouns_score = weat_score(words["top_nouns"], words["bottom_nouns"], innovation_words, tradition_words)
        adjs_score = weat_score(words["top_adjs"], words["bottom_adjs"], innovation_words, tradition_words)

        results.append([model_name, age_val, "Nouns", nouns_score])
        results.append([model_name, age_val, "Adjectives", adjs_score])



In [ ]:

weat_results_df = pd.DataFrame(results, columns=["Model", "Age Group", "POS", "WEAT_IT"])
print(weat_results_df)

In [ ]:
# =========================
# 4. Run WEAT per model × age group
# =========================
results = []

for model_name, age_dict in age_word_sets.items():
    for age_val, words in age_dict.items():
        print(model_name, words , "\n")
        # Get nouns and adjectives for this age group
        nouns_score = weat_score(words["top_nouns"], words["bottom_nouns"], energy_words, experience_words)
        adjs_score = weat_score(words["top_adjs"], words["bottom_adjs"], energy_words, experience_words)

        results.append([model_name, age_val, "Nouns", nouns_score])
        results.append([model_name, age_val, "Adjectives", adjs_score])



In [ ]:
weat_results_df = pd.DataFrame(results, columns=["Model", "Age Group", "POS", "WEAT_EE"])
print(weat_results_df)